In [9]:
import heapq
from typing import List, Tuple, Optional, Set, Dict

State = Tuple[int, ...]

class NPuzzleNode:
    __slots__ = ('state', 'empty_idx', 'g_cost', 'h_cost', 'f_cost', 'parent')

    def __init__(self, state: State, empty_idx: int, g_cost: int, h_cost: int, parent: Optional['NPuzzleNode'] = None):
        self.state = state
        self.empty_idx = empty_idx
        self.g_cost = g_cost
        self.h_cost = h_cost
        self.f_cost = g_cost + h_cost
        self.parent = parent

    def __lt__(self, other: 'NPuzzleNode') -> bool:
        if self.f_cost == other.f_cost:
            return self.h_cost < other.h_cost
        return self.f_cost < other.f_cost

class NPuzzleSolver:
    def __init__(self, initial_matrix: List[List[int]], goal_matrix: List[List[int]]):
        self.size = len(initial_matrix)
        self.initial_state = self._flatten(initial_matrix)
        self.goal_state = self._flatten(goal_matrix)

        self.target_positions: Dict[int, Tuple[int, int]] = {
            val: (i // self.size, i % self.size)
            for i, val in enumerate(self.goal_state) if val != 0
        }

        self.moves = [(-1, 0), (1, 0), (0, -1), (0, 1)]

    def _flatten(self, matrix: List[List[int]]) -> State:
        return tuple(item for row in matrix for item in row)

    def _get_parity(self, state: State) -> int:
        inversions = 0
        state_no_zero = [x for x in state if x != 0]

        for i in range(len(state_no_zero)):
            for j in range(i + 1, len(state_no_zero)):
                if state_no_zero[i] > state_no_zero[j]:
                    inversions += 1

        if self.size % 2 != 0:
            return inversions % 2
        else:
            empty_idx = state.index(0)
            empty_row_from_bottom = self.size - (empty_idx // self.size)
            return (inversions + empty_row_from_bottom) % 2

    def is_solvable(self) -> bool:
        return self._get_parity(self.initial_state) == self._get_parity(self.goal_state)

    def _manhattan_distance(self, state: State) -> int:
        distance = 0
        for i, val in enumerate(state):
            if val != 0:
                current_row, current_col = i // self.size, i % self.size
                target_row, target_col = self.target_positions[val]
                distance += abs(current_row - target_row) + abs(current_col - target_col)
        return distance

    def _get_neighbors(self, node: NPuzzleNode) -> List[NPuzzleNode]:
        neighbors = []
        r, c = node.empty_idx // self.size, node.empty_idx % self.size

        for dr, dc in self.moves:
            new_r, new_c = r + dr, c + dc

            if 0 <= new_r < self.size and 0 <= new_c < self.size:
                new_idx = new_r * self.size + new_c

                new_state_list = list(node.state)
                new_state_list[node.empty_idx], new_state_list[new_idx] = new_state_list[new_idx], new_state_list[node.empty_idx]
                new_state = tuple(new_state_list)

                g_cost = node.g_cost + 1
                h_cost = self._manhattan_distance(new_state)

                neighbors.append(NPuzzleNode(new_state, new_idx, g_cost, h_cost, node))

        return neighbors

    def solve(self) -> Optional[NPuzzleNode]:
        if not self.is_solvable():
            print("Cảnh báo: Không tồn tại lời giải cho bài toán này (Lỗi Parity).")
            return None

        initial_empty_idx = self.initial_state.index(0)
        initial_h = self._manhattan_distance(self.initial_state)
        root = NPuzzleNode(self.initial_state, initial_empty_idx, 0, initial_h)

        open_list: List[NPuzzleNode] = []
        heapq.heappush(open_list, root)

        closed_set: Set[State] = set()

        while open_list:
            current_node = heapq.heappop(open_list)

            if current_node.state == self.goal_state:
                return current_node

            closed_set.add(current_node.state)

            for neighbor in self._get_neighbors(current_node):
                if neighbor.state not in closed_set:
                    heapq.heappush(open_list, neighbor)

        return None

def print_solution(node: Optional[NPuzzleNode], size: int) -> None:
    if not node:
        return

    path = []
    while node:
        path.append(node)
        node = node.parent

    path.reverse()
    print(f"Hoàn thành trong {len(path) - 1} bước di chuyển.\n")

    for step, n in enumerate(path):
        print(f"Bước {step}:")
        for i in range(size):
            row = n.state[i*size : (i+1)*size]
            print("\t".join(str(x) if x != 0 else "_" for x in row))
        print("-" * 20)

if __name__ == "__main__":
    initial_matrix = [
        [1, 2, 3, 4, 5],
        [6, 7, 8, 9, 10],
        [11, 12, 13, 14, 15],
        [16, 17, 18, 0, 19],
        [21, 22, 23, 24, 20]
    ]

    final_matrix = [
        [1, 2, 3, 4, 5],
        [6, 7, 8, 9, 10],
        [11, 12, 13, 14, 15],
        [16, 17, 18, 19, 20],
        [21, 22, 23, 24, 0]
    ]

    solver = NPuzzleSolver(initial_matrix, final_matrix)
    solution = solver.solve()

    if solution:
        print_solution(solution, solver.size)

Hoàn thành trong 2 bước di chuyển.

Bước 0:
1	2	3	4	5
6	7	8	9	10
11	12	13	14	15
16	17	18	_	19
21	22	23	24	20
--------------------
Bước 1:
1	2	3	4	5
6	7	8	9	10
11	12	13	14	15
16	17	18	19	_
21	22	23	24	20
--------------------
Bước 2:
1	2	3	4	5
6	7	8	9	10
11	12	13	14	15
16	17	18	19	20
21	22	23	24	_
--------------------


In [10]:
import heapq
from typing import List, Tuple, Set, FrozenSet, Optional

class TSPNode:
    __slots__ = ('current', 'visited', 'g_cost', 'h_cost', 'f_cost', 'path')

    def __init__(self, current: int, visited: FrozenSet[int], g_cost: float, h_cost: float, path: Tuple[int, ...]):
        self.current = current
        self.visited = visited
        self.g_cost = g_cost
        self.h_cost = h_cost
        self.f_cost = g_cost + h_cost
        self.path = path

    def __lt__(self, other: 'TSPNode') -> bool:
        if self.f_cost == other.f_cost:
            return self.h_cost < other.h_cost
        return self.f_cost < other.f_cost

class TSPSolver:
    def __init__(self, graph: List[List[float]], start_node: int = 0):
        self.graph = graph
        self.num_nodes = len(graph)
        self.start_node = start_node
        self.all_nodes = frozenset(range(self.num_nodes))

    def _mst_cost(self, unvisited: FrozenSet[int]) -> float:
        if not unvisited:
            return 0.0

        unvisited_list = list(unvisited)
        n = len(unvisited_list)
        if n == 1:
            return 0.0

        in_mst = {unvisited_list[0]}
        mst_cost = 0.0
        edges = []

        for i in range(1, n):
            heapq.heappush(edges, (self.graph[unvisited_list[0]][unvisited_list[i]], unvisited_list[0], unvisited_list[i]))

        while len(in_mst) < n and edges:
            cost, u, v = heapq.heappop(edges)
            if v not in in_mst:
                in_mst.add(v)
                mst_cost += cost
                for next_v in unvisited_list:
                    if next_v not in in_mst:
                        heapq.heappush(edges, (self.graph[v][next_v], v, next_v))

        return mst_cost

    def _heuristic(self, current: int, unvisited: FrozenSet[int]) -> float:
        if not unvisited:
            return self.graph[current][self.start_node]

        mst_weight = self._mst_cost(unvisited)

        min_to_unvisited = min(self.graph[current][u] for u in unvisited)
        min_from_unvisited_to_start = min(self.graph[u][self.start_node] for u in unvisited)

        return mst_weight + min_to_unvisited + min_from_unvisited_to_start

    def solve(self) -> Optional[Tuple[Tuple[int, ...], float]]:
        initial_visited = frozenset([self.start_node])
        initial_unvisited = self.all_nodes - initial_visited
        initial_h = self._heuristic(self.start_node, initial_unvisited)

        root = TSPNode(self.start_node, initial_visited, 0.0, initial_h, (self.start_node,))

        pq: List[TSPNode] = []
        heapq.heappush(pq, root)

        closed_set: Set[Tuple[int, FrozenSet[int]]] = set()

        while pq:
            node = heapq.heappop(pq)
            state = (node.current, node.visited)

            if len(node.visited) == self.num_nodes:
                final_cost = node.g_cost + self.graph[node.current][self.start_node]
                final_path = node.path + (self.start_node,)
                return final_path, final_cost

            if state in closed_set:
                continue
            closed_set.add(state)

            unvisited = self.all_nodes - node.visited
            for next_node in unvisited:
                new_visited = node.visited | frozenset([next_node])
                new_unvisited = unvisited - frozenset([next_node])

                g_cost = node.g_cost + self.graph[node.current][next_node]
                h_cost = self._heuristic(next_node, new_unvisited)
                new_path = node.path + (next_node,)

                child = TSPNode(next_node, new_visited, g_cost, h_cost, new_path)
                heapq.heappush(pq, child)

        return None

if __name__ == "__main__":
    distance_matrix = [
        [0, 10, 15, 20],
        [10, 0, 35, 25],
        [15, 35, 0, 30],
        [20, 25, 30, 0]
    ]

    solver = TSPSolver(distance_matrix)
    result = solver.solve()

    if result:
        path, cost = result
        print(f"Path: {' -> '.join(map(str, path))}")
        print(f"Cost: {cost}")
    else:
        print("No solution found.")

Path: 0 -> 2 -> 3 -> 1 -> 0
Cost: 80.0
